In [6]:
import pandas as pd
import pulp as plp
solver = plp.GLPK_CMD()


# Lecture des fichiers comportant les données
aliment_df = pd.read_csv('../data/aliments.csv', index_col=0)
besoin_df = pd.read_csv('../data/besoins.csv', index_col = 0)

# Construction des données
aliments = list(aliment_df.index)
nutriments = [col for col in aliment_df.columns if col != 'cout']
aliment_df[nutriments] = aliment_df[nutriments] / 100
aliment_df['cout'] = aliment_df['cout'] / 100
besoins = besoin_df['minimum'].to_dict()
cout = {i : aliment_df.loc[i, 'cout'] for i in aliments}

# Problème de minimisation
model = plp.LpProblem('Minisation_du_cout', plp.LpMinimize)

# Variables de décision
x = plp.LpVariable.dicts('x',  cout.keys(), lowBound=0, upBound = 300, cat='continuous')

# Fonction objectif
model += plp.lpSum([cout[i]*x[i] for i in aliments])

# Contraintes
for j in nutriments:
    model += plp.lpSum([aliment_df.loc[i, j]*x[i] for i in aliments]) >= besoins[j]

# Résolution
model.solve(solver)

# Résultat
print("Status : ", plp.LpStatus[model.status])
for i, var in x.items(): 
    if var.value() > 0:
        print(f"{i} : {var.value():.1f}g") 
print(f"Coût Total : {plp.value(model.objective):.2f}€")

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /tmp/e8d42494d14f497b8efa8c41a2ac14eb-pulp.lp -o /tmp/e8d42494d14f497b8efa8c41a2ac14eb-pulp.out
 -w /tmp/e8d42494d14f497b8efa8c41a2ac14eb-pulp.sol
Reading problem data from '/tmp/e8d42494d14f497b8efa8c41a2ac14eb-pulp.lp'...
4 rows, 8 columns, 25 non-zeros
24 lines were read
GLPK Simplex Optimizer 5.0
4 rows, 8 columns, 25 non-zeros
Preprocessing...
4 rows, 8 columns, 25 non-zeros
Scaling...
 A: min|aij| =  1.000e-02  max|aij| =  8.840e+00  ratio =  8.840e+02
GM: min|aij| =  2.856e-01  max|aij| =  3.502e+00  ratio =  1.226e+01
EQ: min|aij| =  8.156e-02  max|aij| =  1.000e+00  ratio =  1.226e+01
Constructing initial basis...
Size of triangular part is 4
      0: obj =   0.000000000e+00 inf =   5.838e+03 (4)
      8: obj =   4.207307692e+00 inf =   0.000e+00 (0)
*    10: obj =   2.186914604e+00 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Time used:   0.0 secs
Memory used: 0.0 Mb (39693 bytes)
Writing b